# 05 — Gold Layer: fact_enrolment (Transaction Fact)

**Purpose:** Build the enrolment transaction fact table from Silver.

**Kimball fact type:** Transaction
- One row per confirmed enrolment event
- Inserted once — never updated (transaction facts are immutable)
- If enrolment status changes (Active → Withdrawn), the existing row is updated
  via MERGE because status is an attribute of the enrolment, not a new event

**Measures:**
- `enrolment_fee` — gross tuition fee
- `scholarship_amount` — discount applied
- `net_fee_payable` — pre-calculated in Silver (enrolment_fee - scholarship_amount)
- `credit_points` — academic load
- `is_withdrawal` — 1 if student withdrew
- `is_completion` — 1 if student completed

## Parameters

In [ ]:
pipeline_run_date = "2024-01-15"

## Setup

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType
from delta.tables import DeltaTable

run_date = pipeline_run_date
FACT_ENROLMENT = "gold.fact_enrolment"


def table_exists(table_name):
    try:
        spark.table(table_name)
        return True
    except Exception:
        return False


def date_to_key(date_col):
    return F.when(
        date_col.isNotNull(),
        F.date_format(date_col, "yyyyMMdd").cast(IntegerType())
    ).otherwise(F.lit(0))


print(f"Pipeline run date: {run_date}")
print("Setup complete.")

## Step 1 — Build Incoming Enrolment Fact

Join `silver.enrolment` to dimension tables to resolve surrogate keys.
Use point-in-time join for `dim_student` — match the student version
that was current at the time of enrolment.

In [ ]:
silver_enrol = spark.table("silver.enrolment")

# Point-in-time join to dim_student
# Match the student version active at enrolment_date
dim_student = spark.table("gold.dim_student").select(
    "student_sk", "student_id", "_valid_from", "_valid_to"
)
dim_course  = spark.table("gold.dim_course").select("course_sk", "course_id")
dim_campus  = spark.table("gold.dim_campus").select("campus_sk", "campus_id")
dim_intake  = spark.table("gold.dim_intake").select("intake_sk", "intake_id")
dim_status  = spark.table("gold.dim_status").filter(
    F.col("status_category") == "Enrolment"
).select("status_sk", "status_code")

incoming_enrolments = (
    silver_enrol
    # Point-in-time join: student version active at enrolment_date
    .join(
        dim_student,
        (silver_enrol.student_id == dim_student.student_id) &
        (silver_enrol.enrolment_date >= dim_student._valid_from) &
        (silver_enrol.enrolment_date < dim_student._valid_to),
        how="left"
    )
    .join(dim_course,  silver_enrol.course_code == dim_course.course_id,   how="left")
    .join(dim_campus,  silver_enrol.campus_code == dim_campus.campus_id,   how="left")
    .join(dim_intake,  silver_enrol.intake_code == dim_intake.intake_id,   how="left")
    .join(dim_status,  silver_enrol.enrolment_status == dim_status.status_code, how="left")

    .withColumn("enrolment_date_key", date_to_key(F.col("enrolment_date")))
    .withColumn("is_withdrawal",
        F.when(F.col("enrolment_status") == "Withdrawn", F.lit(1)).otherwise(F.lit(0)))
    .withColumn("is_completion",
        F.when(F.col("enrolment_status") == "Completed", F.lit(1)).otherwise(F.lit(0)))

    .select(
        silver_enrol.enrolment_id,
        F.coalesce(dim_student.student_sk, F.lit(-1)).alias("student_sk"),
        F.coalesce(dim_course.course_sk,   F.lit(-1)).alias("course_sk"),
        F.coalesce(dim_campus.campus_sk,   F.lit(-1)).alias("campus_sk"),
        F.coalesce(dim_intake.intake_sk,   F.lit(-1)).alias("intake_sk"),
        F.coalesce(dim_status.status_sk,   F.lit(-1)).alias("status_sk"),
        F.col("enrolment_date_key"),
        silver_enrol.enrolment_fee,
        silver_enrol.scholarship_amount,
        silver_enrol.net_fee_payable,
        silver_enrol.credit_points,
        F.col("is_withdrawal"),
        F.col("is_completion"),
        F.current_timestamp().alias("_loaded_at"),
    )
)

print(f"Incoming enrolment rows: {incoming_enrolments.count()}")
incoming_enrolments.show(truncate=False)

## Step 2 — MERGE into fact_enrolment

MERGE on `enrolment_id`.
New enrolments are inserted. Existing enrolments are updated if status changed
(e.g. Active → Withdrawn — updates `is_withdrawal`, `status_sk`).

In [ ]:
if not table_exists(FACT_ENROLMENT):
    print("[fact_enrolment] First run — creating table")
    w = Window.orderBy("enrolment_id")
    first_load = incoming_enrolments.withColumn(
        "enrolment_sk", F.row_number().over(w).cast(IntegerType())
    )
    first_load.write.format("delta").mode("overwrite").saveAsTable(FACT_ENROLMENT)
    print(f"[fact_enrolment] Inserted {first_load.count()} rows")

else:
    print("[fact_enrolment] Merging into existing table")

    existing = spark.table(FACT_ENROLMENT)
    max_sk = existing.agg(F.max("enrolment_sk")).collect()[0][0] or 0

    new_enrolments = incoming_enrolments.join(
        existing.select("enrolment_id"), on="enrolment_id", how="left_anti"
    )
    new_count = new_enrolments.count()

    if new_count > 0:
        w = Window.orderBy("enrolment_id")
        new_enrolments = new_enrolments.withColumn(
            "enrolment_sk",
            (F.row_number().over(w) + F.lit(max_sk)).cast(IntegerType())
        )
        new_enrolments.write.format("delta").mode("append").saveAsTable(FACT_ENROLMENT)

    # Update status-related columns for existing enrolments
    DeltaTable.forName(spark, FACT_ENROLMENT).alias("t").merge(
        incoming_enrolments.alias("s"),
        "t.enrolment_id = s.enrolment_id"
    ).whenMatchedUpdate(set={
        "status_sk":    "s.status_sk",
        "is_withdrawal": "s.is_withdrawal",
        "is_completion": "s.is_completion",
        "_loaded_at":   "current_timestamp()",
    }).execute()

    print(f"[fact_enrolment] New rows: {new_count} | Status updates applied")

total = spark.table(FACT_ENROLMENT).count()
print(f"[fact_enrolment] Total rows: {total}")

## Verification

In [ ]:
fact = spark.table(FACT_ENROLMENT)

print("Revenue summary:")
fact.agg(
    F.count("enrolment_sk").alias("total_enrolments"),
    F.sum("enrolment_fee").alias("gross_revenue"),
    F.sum("scholarship_amount").alias("total_scholarships"),
    F.sum("net_fee_payable").alias("net_revenue"),
    F.sum("is_withdrawal").alias("withdrawals"),
    F.sum("is_completion").alias("completions"),
).show(truncate=False)

print("\nAll enrolment rows:")
fact.select(
    "enrolment_sk", "enrolment_id", "student_sk",
    "enrolment_fee", "scholarship_amount", "net_fee_payable",
    "credit_points", "is_withdrawal", "is_completion"
).show(truncate=False)

# Verify net_fee_payable = enrolment_fee - scholarship_amount
mismatch = fact.filter(
    F.round(F.col("net_fee_payable"), 2) !=
    F.round(F.col("enrolment_fee") - F.col("scholarship_amount"), 2)
).count()
print(f"net_fee_payable mismatch rows (should be 0): {mismatch}")

## Summary

`fact_enrolment` built and verified.
6 enrolment rows — one per enrolled student.
Revenue, scholarship, credit point, and status measures all correct.

**Next step:** Run `06_gold_fact_attendance.ipynb`.